# Quantum Kernel Teacher Defect Task

This notebook builds a **quantum-tailored image classification task**. The images are still tiny grayscale arrays, but the label is not created by inserting an obvious dark blob. Instead, a `ZZFeatureMap` quantum kernel acts as a teacher: points are labeled by their similarity to hidden centers in quantum feature space.

This is the most defensible hackathon direction if you want to talk about a classically hard problem. It does **not** prove quantum advantage on real medical data, but it gives you a clean experimental story:

1. The blob dataset is a sanity baseline and is classically easy.
2. A quantum feature map can define a label geometry that common classical baselines do not naturally see.
3. A `QSVC` with the matching quantum kernel is the first method to try before training a variational QNN.

## 1. Imports

The defaults are intentionally small. Quantum kernel evaluation scales with the number of train/test pairs, so increase sample counts only after the first run works.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, balanced_accuracy_score, classification_report, f1_score
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import SVC

from qiskit.circuit.library import n_local, zz_feature_map
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit_algorithms.optimizers import COBYLA, SPSA
from qiskit_machine_learning.algorithms import QSVC
from qiskit_machine_learning.algorithms.classifiers import NeuralNetworkClassifier
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.neural_networks import SamplerQNN

SEED = 8398
np.random.seed(SEED)

## 2. Helpers: Metrics, Synthetic Images, and Quantum Teacher

The helper `make_quantum_teacher_dataset` samples latent vectors, turns them into texture images, then labels them using a quantum kernel decision rule. The texture image is a carrier for the latent features; the target is the global quantum-kernel geometry.

In [ ]:
def evaluate_predictions(name, y_true, y_pred, *, show_confusion=True):
    """Print metrics that are useful for balanced binary classification."""
    print(f"\n{name}")
    print("balanced accuracy:", balanced_accuracy_score(y_true, y_pred))
    print("defect F1 score  :", f1_score(y_true, y_pred, zero_division=0))
    print(classification_report(y_true, y_pred, target_names=["clean", "defect"], zero_division=0))

    if show_confusion:
        ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=["clean", "defect"])
        plt.title(name)
        plt.show()


def evaluate_classifier(name, model, X_eval, y_eval, *, show_confusion=True):
    y_pred = model.predict(X_eval)
    evaluate_predictions(name, y_eval, y_pred, show_confusion=show_confusion)
    return y_pred


def balanced_subset(X_data, y_data, n_per_class=24, seed=SEED):
    """Make a small balanced set for slower quantum experiments."""
    rng = np.random.default_rng(seed)
    chosen = []
    for cls in np.unique(y_data):
        cls_idx = np.flatnonzero(y_data == cls)
        take = min(n_per_class, len(cls_idx))
        chosen.extend(rng.choice(cls_idx, size=take, replace=False).tolist())
    rng.shuffle(chosen)
    return X_data[chosen], y_data[chosen]


def latent_to_texture_images(X_latent, size=8):
    """Convert low-dimensional latent vectors into small grayscale texture images.

    The image is only a visualization/carrier for the latent features. The label is
    not created by inserting a visible blob; it is created by a quantum-kernel teacher.
    """
    grid = np.linspace(-1.0, 1.0, size)
    rr, cc = np.meshgrid(grid, grid, indexing="ij")
    images = []

    for x in X_latent:
        terms = [
            np.sin(np.pi * rr + x[0]),
            np.cos(np.pi * cc + x[1]),
            np.sin(np.pi * (rr + cc) + x[2]),
            np.cos(np.pi * (rr - cc) + x[3]),
        ]
        if len(x) > 4:
            terms.append(np.sin(np.pi * rr * cc + x[4]))
        if len(x) > 5:
            terms.append(np.cos(np.pi * (rr**2 - cc**2) + x[5]))

        texture = np.mean(terms, axis=0)
        texture = (texture - texture.min()) / (texture.max() - texture.min() + 1e-12)
        texture = 0.25 + 0.55 * texture
        images.append(texture)

    return np.array(images)


def make_quantum_teacher_dataset(
    *,
    n_samples=100,
    n_qubits=5,
    n_centers=12,
    feature_reps=1,
    image_size=8,
    seed=SEED,
):
    """Create a synthetic image dataset whose labels are generated in quantum feature space.

    A few random centers act like support vectors. The label is the sign of a weighted
    sum of quantum kernel similarities to those centers. This is an engineered task,
    not a medical claim, but it is much closer to the theory behind quantum kernels.
    """
    rng = np.random.default_rng(seed)
    X_latent = rng.uniform(0.0, 2 * np.pi, size=(n_samples, n_qubits))
    centers = rng.uniform(0.0, 2 * np.pi, size=(n_centers, n_qubits))

    feature_map = zz_feature_map(feature_dimension=n_qubits, reps=feature_reps)
    quantum_kernel = FidelityQuantumKernel(feature_map=feature_map)

    center_kernel = quantum_kernel.evaluate(x_vec=X_latent, y_vec=centers)
    weights = rng.normal(size=n_centers)
    raw_scores = center_kernel @ weights
    threshold = np.median(raw_scores)
    labels = (raw_scores > threshold).astype(int)

    images = latent_to_texture_images(X_latent, size=image_size)

    return {
        "X_latent": X_latent,
        "images": images,
        "labels": labels,
        "centers": centers,
        "weights": weights,
        "scores": raw_scores,
        "threshold": threshold,
        "feature_map": feature_map,
        "quantum_kernel": quantum_kernel,
    }


def show_labeled_textures(images, labels, scores=None, n_show=12, cols=6):
    order = np.arange(min(n_show, len(images)))
    rows = int(np.ceil(len(order) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.5, rows * 1.7))
    axes = np.ravel(axes)

    for ax, idx in zip(axes, order):
        ax.imshow(images[idx], cmap="gray_r", vmin=0, vmax=1)
        label = "DEFECT" if labels[idx] else "clean"
        suffix = "" if scores is None else f"\nscore={scores[idx]:.3f}"
        ax.set_title(f"#{idx} {label}{suffix}", fontsize=8, color="crimson" if labels[idx] else "black")
        ax.set_xticks([])
        ax.set_yticks([])

    for ax in axes[len(order):]:
        ax.axis("off")

    fig.tight_layout()
    plt.show()


def centered_kernel_alignment(K, y):
    """Centered alignment between a kernel matrix and the ideal label kernel y y^T."""
    y_signed = 2 * np.asarray(y) - 1
    Y = np.outer(y_signed, y_signed)
    n = K.shape[0]
    H = np.eye(n) - np.ones((n, n)) / n
    Kc = H @ K @ H
    Yc = H @ Y @ H
    denom = np.linalg.norm(Kc, "fro") * np.linalg.norm(Yc, "fro")
    return float(np.sum(Kc * Yc) / denom) if denom else np.nan

## 3. Generate a Quantum-Tailored Dataset

`N_QUBITS=5` is a reasonable hackathon default. It is large enough that the feature map is nontrivial, but small enough that statevector-backed kernels run locally. The fixed `DATA_SEED` gives one reproducible benchmark; change it later to test robustness.

In [ ]:
N_SAMPLES = 100
N_QUBITS = 5
N_CENTERS = 12
FEATURE_REPS = 1
IMAGE_SIZE = 8
DATA_SEED = SEED + 501

data = make_quantum_teacher_dataset(
    n_samples=N_SAMPLES,
    n_qubits=N_QUBITS,
    n_centers=N_CENTERS,
    feature_reps=FEATURE_REPS,
    image_size=IMAGE_SIZE,
    seed=DATA_SEED,
)

X_latent = data["X_latent"]
images = data["images"]
y = data["labels"]
feature_map = data["feature_map"]
quantum_kernel = data["quantum_kernel"]

print("latent shape:", X_latent.shape)
print("image shape :", images.shape)
print("class balance:", {"clean": int((y == 0).sum()), "defect": int((y == 1).sum())})
print("feature map qubits:", feature_map.num_qubits)
print("feature map depth :", feature_map.decompose().depth())

## 4. Visual Check

The labels should not be obvious from a single dark pixel. That is the point: the task is meant to be global and feature-map-shaped.

In [ ]:
show_labeled_textures(images, y, scores=data["scores"], n_show=12, cols=6)

## 5. Train/Test Split

We keep both views:

- `X_latent`: the compact feature vector that the quantum circuit receives.
- `X_pixels`: flattened texture images for classical image baselines.

In [ ]:
X_pixels = images.reshape(len(images), -1)

X_latent_train, X_latent_test, X_pixels_train, X_pixels_test, y_train, y_test = train_test_split(
    X_latent,
    X_pixels,
    y,
    test_size=0.30,
    random_state=SEED,
    stratify=y,
)

print("train:", X_latent_train.shape, "clean:", int((y_train == 0).sum()), "defect:", int((y_train == 1).sum()))
print("test :", X_latent_test.shape, "clean:", int((y_test == 0).sum()), "defect:", int((y_test == 1).sum()))

## 6. Classical Baselines

A strong claim needs strong baselines. These are not exhaustive, but they catch the easy wins: linear decision surfaces, smooth RBF similarity, polynomial interactions, and random forests.

In [ ]:
classical_latent_models = {
    "Latent Logistic Regression": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)),
    ]),
    "Latent RBF SVM": Pipeline([
        ("scale", StandardScaler()),
        ("model", SVC(kernel="rbf", C=1.0, class_weight="balanced", random_state=SEED)),
    ]),
    "Latent Polynomial SVM degree 3": Pipeline([
        ("scale", StandardScaler()),
        ("model", SVC(kernel="poly", degree=3, C=1.0, class_weight="balanced", random_state=SEED)),
    ]),
    "Latent Random Forest": RandomForestClassifier(
        n_estimators=250,
        class_weight="balanced",
        random_state=SEED,
    ),
}

for name, model in classical_latent_models.items():
    model.fit(X_latent_train, y_train)
    evaluate_classifier(name, model, X_latent_test, y_test, show_confusion=False)

## 7. Classical Image Baselines

These baselines only see the flattened image. PCA keeps the setup close to your original notebook, where the quantum circuit receives a small feature vector.

In [ ]:
N_PCA = N_QUBITS

pixel_scaler = StandardScaler()
X_pixels_train_scaled = pixel_scaler.fit_transform(X_pixels_train)
X_pixels_test_scaled = pixel_scaler.transform(X_pixels_test)

pca = PCA(n_components=N_PCA, random_state=SEED)
X_pixels_train_pca = pca.fit_transform(X_pixels_train_scaled)
X_pixels_test_pca = pca.transform(X_pixels_test_scaled)

print("PCA explained variance:", pca.explained_variance_ratio_.sum())

pixel_models = {
    "Pixel PCA Logistic Regression": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)),
    ]),
    "Pixel PCA RBF SVM": Pipeline([
        ("scale", StandardScaler()),
        ("model", SVC(kernel="rbf", class_weight="balanced", random_state=SEED)),
    ]),
    "Raw Pixel Random Forest": RandomForestClassifier(
        n_estimators=250,
        class_weight="balanced",
        random_state=SEED,
    ),
}

pixel_models["Pixel PCA Logistic Regression"].fit(X_pixels_train_pca, y_train)
evaluate_classifier("Pixel PCA Logistic Regression", pixel_models["Pixel PCA Logistic Regression"], X_pixels_test_pca, y_test, show_confusion=False)

pixel_models["Pixel PCA RBF SVM"].fit(X_pixels_train_pca, y_train)
evaluate_classifier("Pixel PCA RBF SVM", pixel_models["Pixel PCA RBF SVM"], X_pixels_test_pca, y_test, show_confusion=False)

pixel_models["Raw Pixel Random Forest"].fit(X_pixels_train, y_train)
evaluate_classifier("Raw Pixel Random Forest", pixel_models["Raw Pixel Random Forest"], X_pixels_test, y_test, show_confusion=False)

## 8. Quantum Kernel Classifier (`QSVC`)

This is the main recommendation. The optimization is convex on the SVM side, and the quantum part is isolated to kernel estimation. For a hackathon, that makes it more stable than a variational QNN.

In [ ]:
KERNEL_TRAIN_PER_CLASS = 30

X_kernel_train, y_kernel_train = balanced_subset(
    X_latent_train,
    y_train,
    n_per_class=KERNEL_TRAIN_PER_CLASS,
)

qsvc = QSVC(quantum_kernel=quantum_kernel, C=1.0, class_weight="balanced")
qsvc.fit(X_kernel_train, y_kernel_train)

evaluate_classifier("Quantum Kernel QSVC", qsvc, X_latent_test, y_test)

## 9. Kernel Alignment Diagnostics

This is a compact way to ask whether a kernel geometry agrees with the labels. Higher centered alignment means the kernel looks more like the ideal label kernel.

In [ ]:
X_align, y_align = balanced_subset(X_latent_train, y_train, n_per_class=24)

K_quantum = quantum_kernel.evaluate(x_vec=X_align)
X_align_scaled = StandardScaler().fit_transform(X_align)
K_rbf = rbf_kernel(X_align_scaled, gamma=1 / X_align_scaled.shape[1])

print("quantum kernel alignment:", centered_kernel_alignment(K_quantum, y_align))
print("classical RBF alignment  :", centered_kernel_alignment(K_rbf, y_align))
print("quantum off-diagonal variance:", np.var(K_quantum[~np.eye(len(K_quantum), dtype=bool)]))
print("RBF off-diagonal variance    :", np.var(K_rbf[~np.eye(len(K_rbf), dtype=bool)]))

## 10. What to Tune Next

- Increase `N_QUBITS` from `5` to `6` only after the first run works.
- Try `FEATURE_REPS=2`, but expect a slower kernel and possible concentration.
- Increase `KERNEL_TRAIN_PER_CLASS` for better QSVC performance.
- Add stronger classical baselines if you want a more serious claim: tuned RBF grids, polynomial grids, gradient boosting, and a tiny CNN.
- The honest phrasing is "quantum-tailored benchmark with a plausible quantum-kernel advantage path," not "medical quantum advantage."

## References and AI Tools Disclosure

This notebook was drafted with OpenAI Codex/GPT-5 assistance in the local hackathon workspace. The team should treat the code and results as AI-assisted prototypes and verify claims before presenting them.

AI-assisted notebooks in this `main_challenge` folder:

- `01_defect_classical_hardness_audit.ipynb`
- `02_quantum_kernel_teacher_defects.ipynb`
- `03_projected_quantum_kernel_features.ipynb`
- `04_qnn_vs_kernel_bakeoff.ipynb`
- `05_qcnn_industrial_microdefects.ipynb`
- `06_data_reuploading_qnn_microdefects.ipynb`
- `07_qnn_kernel_pivot_scoreboard.ipynb`
- `08_imbalanced_rare_defect_qnn.ipynb`
- `09_normal_only_anomaly_detection.ipynb`
- `10_heat_exchanger_network_qubo_qaoa.ipynb`
- `11_Final_QCNN_rare_defect_detection.ipynb`

References used for the quantum-ML and optimization direction:

- Cong, Choi, and Lukin, "Quantum convolutional neural networks," Nature Physics 15, 1273-1278 (2019): https://www.nature.com/articles/s41567-019-0648-8
- Perez-Salinas et al., "Data re-uploading for a universal quantum classifier," Quantum 4, 226 (2020): https://doi.org/10.22331/q-2020-02-06-226 and https://arxiv.org/abs/1907.02085
- McClean et al., "Barren plateaus in quantum neural network training landscapes," Nature Communications 9, 4812 (2018): https://www.nature.com/articles/s41467-018-07090-4
- Havlicek et al., "Supervised learning with quantum-enhanced feature spaces," Nature 567, 209-212 (2019): https://www.nature.com/articles/s41586-019-0980-2
- Scholkopf et al., "Estimating the Support of a High-Dimensional Distribution," Neural Computation 13(7), 1443-1471 (2001): https://doi.org/10.1162/089976601750264965
- Furman and Sahinidis, "Computational complexity of heat exchanger network synthesis," Computers & Chemical Engineering 25(9-10), 1371-1390 (2001): https://doi.org/10.1016/S0098-1354(01)00681-0
- Qiskit Machine Learning `SamplerQNN` documentation: https://qiskit-community.github.io/qiskit-machine-learning/stubs/qiskit_machine_learning.neural_networks.SamplerQNN.html